# Integrating with LangGraph

Agents are where LLM failures turn into actions: an email sent, a refund issued, a file deleted. LangGraph agents call
their models through LangChain chat models, so `immune.init()` screens every step of the graph: each model call, each
tool result the model reads, and each tool call it makes.

| You have | Section |
| --- | --- |
| A tool-calling agent built with `StateGraph` | [1](#1.-A-support-agent) |
| Multi-turn agents with a checkpointer | [2](#2.-Confirmations-across-turns) |
| Several LLM nodes with different jobs | [3](#3.-One-site-per-node) |
| A way to test the agent against a real incident | [4](#4.-Rehearse-an-incident-through-your-graph) |

**Requirements:** `pip install immune-ai langgraph langchain-openai`, with `OPENAI_API_KEY` and `TYPESAFE_API_KEY`
set. `langchain.agents.create_agent` builds a LangGraph graph too, and is covered the same way.

```
            ┌──────────── your graph ────────────┐
 user ──►   │  agent node ──► tools node ──┐     │
            │      ▲                       │     │
            │      └───────────────────────┘     │
            └──────┬─────────────────────────────┘
                   │ every model call
                   ▼
                Immune: input · tool results (data) · tool calls · reply
```

## 1. A support agent

**Before.** A standard LangGraph agent with two tools:

In [1]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

ORDERS = {"5521": {"item": "Ridgeline 2P tent", "total": 249.00, "status": "delivered"}}


@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its number."""
    return ORDERS.get(order_id, {"error": "no such order"})


@tool
def issue_refund(order_id: str, amount: float) -> dict:
    """Refund an order, in full or in part."""
    return {"order_id": order_id, "refunded": amount}


TOOLS = [lookup_order, issue_refund]
model = ChatOpenAI(model="gpt-5.4-mini").bind_tools(TOOLS)
SYSTEM = SystemMessage("You are the support agent for Pinecrest Outfitters. Use the tools to help customers.")


def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke([SYSTEM, *state["messages"]])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
app = graph.compile()

result = app.invoke({"messages": [("user", "Order 5521: my tent arrived with a broken pole. What are my options?")]})
print(result["messages"][-1].content)

I’m sorry about the broken pole on your Ridgeline 2P tent.

Your order was delivered, so the usual options are:

1. **Replacement part** – if we can source the pole or a repair kit, that’s often the fastest fix.
2. **Return for refund** – if the tent is unusable or you’d prefer not to keep it.
3. **Partial refund** – if you’d like to keep the tent and get compensation for the damaged part.
4. **Full refund/replacement review** – for transit damage or a manufacturing issue, we can review the case.

If you’d like, I can help you choose the best option. The next step is usually to confirm whether you want:
- a **replacement pole/repair**,  
- a **partial refund**, or  
- a **full refund**.


**After.** Immune needs three things, none of which touch the graph:

```diff
+ import immune
+ immune.init(config={"sites": {"support-agent": {"user_facing": True,
+     "tools": {"issue_refund": {"writes_state": True, "irreversible": True}}}}})
  ...
+ with immune.site("support-agent"), immune.session("thread-5521"):
      result = app.invoke(...)
```

- **Tool capabilities** tell Immune which tools send data out, change state or can't be undone.
- **`user_facing: True`** means a person is in the loop, so risky calls become confirmation questions instead of
  silent holds.
- **The session** should be your conversation or thread id, so Immune can follow the agent across turns.

Every `AIMessage` in the result has its own verdict: `immune.verdict(message)`.

In [2]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import immune

immune.init(
    config={
        "sites": {
            "support-agent": {
                "user_facing": True,
                "tools": {"issue_refund": {"writes_state": True, "irreversible": True}},
            }
        }
    }
)
ORDERS = {"5521": {"item": "Ridgeline 2P tent", "total": 249.00, "status": "delivered"}}


@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its number."""
    return ORDERS.get(order_id, {"error": "no such order"})


@tool
def issue_refund(order_id: str, amount: float) -> dict:
    """Refund an order, in full or in part."""
    return {"order_id": order_id, "refunded": amount}


TOOLS = [lookup_order, issue_refund]
model = ChatOpenAI(model="gpt-5.4-mini").bind_tools(TOOLS)
SYSTEM = SystemMessage("You are the support agent for Pinecrest Outfitters. Use the tools to help customers.")


def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke([SYSTEM, *state["messages"]])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
app = graph.compile()

with immune.site("support-agent"), immune.session("thread-5521"):
    result = app.invoke(
        {"messages": [("user", "Order 5521: my tent arrived with a broken pole. What are my options?")]}
    )

for message in result["messages"]:
    if message.type == "ai":
        calls = [f"{call['name']}({call['args']})" for call in message.tool_calls]
        print(f"[{immune.verdict(message).action}] model -> {', '.join(calls) or message.content[:120]}")
    elif message.type == "tool":
        print(f"         tool  <- {message.content[:120]}")

[allow] model -> lookup_order({'order_id': '5521'})
         tool  <- {"item": "Ridgeline 2P tent", "total": 249.0, "status": "delivered"}
[allow] model -> Sorry to hear that your tent arrived with a broken pole.

For order 5521, I can help with these options:
- Replacement/r


**What you see:** the same agent run, with a verdict for every model call. Nothing in the graph changed: the
checks happen below LangChain, on each HTTP call to the model.

## 2. Confirmations across turns

Some actions should wait for the customer to say yes. Here a small rule (a *vaccine*, see
[Vaccines](12_vaccines.ipynb)) says refunds over $100 need confirmation. Immune removes the refund call from the
model's reply and asks the customer instead. When the customer says yes, the same call, sealed to this conversation,
goes through. The graph uses a checkpointer, as multi-turn LangGraph apps do; the Immune session is the thread id.

In [3]:
import tempfile
from pathlib import Path

from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import immune

# In your project this file lives in a vaccines/ folder next to immune.yaml.
rule = Path(tempfile.mkdtemp()) / "refund_over_limit.yaml"
rule.write_text("""
id: pinecrest.refund_over_limit
title: Refunds over $100 need the customer's confirmation
stage: tool
detect:
  tool: issue_refund
  argument: {path: amount, greater_than: 100}
respond:
  tool: confirm
enforcement: enforce
""")
immune.init(
    vaccines=[rule],
    config={"sites": {"support-agent": {"user_facing": True, "tools": {"issue_refund": {"writes_state": True}}}}},
)
refunds: list[dict] = []


@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its number."""
    return {"order_id": order_id, "item": "Ridgeline 2P tent", "total": 249.00, "status": "delivered"}


@tool
def issue_refund(order_id: str, amount: float) -> dict:
    """Refund an order, in full or in part."""
    refunds.append({"order_id": order_id, "amount": amount})
    return {"order_id": order_id, "refunded": amount}


TOOLS = [lookup_order, issue_refund]
model = ChatOpenAI(model="gpt-5.4-mini").bind_tools(TOOLS)
SYSTEM = SystemMessage(
    "You are the support agent for Pinecrest Outfitters. Refund damaged items in full when the customer asks."
)


def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke([SYSTEM, *state["messages"]])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
app = graph.compile(checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "thread-7788"}}

for text in ["Order 5521 arrived with a broken pole. Please refund the full $249.", "Yes, please go ahead."]:
    with immune.site("support-agent"), immune.session("thread-7788"):
        result = app.invoke({"messages": [("user", text)]}, thread)
    print(f"customer: {text}\nagent:    {result['messages'][-1].content}\nrefunds:  {refunds}\n")

customer: Order 5521 arrived with a broken pole. Please refund the full $249.
agent:    Before I do that, please confirm: issue_refund(amount=249, order_id='5521'). Reply "yes" to continue. (ref 95e8968961)
refunds:  []



customer: Yes, please go ahead.
agent:    Done — I’ve refunded the full $249 for order 5521.
refunds:  [{'order_id': '5521', 'amount': 249.0}]



**What you see:** on the first turn the refund is held and the agent's reply asks the customer to confirm. On the
second, the customer's yes releases exactly that call, and the refund runs once. The confirmation is sealed to the
call and the conversation, so a "yes" planted in a document, or a different amount, unlocks nothing.

## 3. One site per node

Different nodes do different jobs, and deserve different postures. Name a site inside each node function. Here a
triage node classifies the message with structured output, and only the responder enforces the off-topic detector:

In [4]:
from typing import Literal

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from pydantic import BaseModel

import immune

immune.init(config={"sites": {"responder": {"enforce": ["input.off_task"]}}})


class Intent(BaseModel):
    topic: Literal["order", "product", "other"]


class State(MessagesState):
    topic: str


triage_model = ChatOpenAI(model="gpt-5.4-mini").with_structured_output(Intent)
reply_model = ChatOpenAI(model="gpt-5.4-mini")


def triage(state: State) -> dict:
    with immune.site("triage"):
        return {"topic": triage_model.invoke(state["messages"]).topic}


def respond(state: State) -> dict:
    with immune.site("responder"):
        prompt = [
            (
                "system",
                "You are the support assistant for Pinecrest Outfitters, a camping store. "
                "Two-person tents: Ridgeline 2P (1.9 kg, $249) and Summit Lite 2P (1.3 kg, $329).",
            )
        ]
        return {"messages": [reply_model.invoke([*prompt, *state["messages"]])]}


graph = StateGraph(State)
graph.add_node("triage", triage)
graph.add_node("respond", respond)
graph.add_edge(START, "triage")
graph.add_edge("triage", "respond")
graph.add_edge("respond", END)
app = graph.compile()

for question in ["Which of your tents is lightest?", "Can you write my history essay about Rome?"]:
    result = app.invoke({"messages": [("user", question)]})
    reply = result["messages"][-1]
    print(f"[{result['topic']}] {question}\n  -> {reply.content[:160]}\n  verdict: {immune.verdict(reply).action}\n")

[product] Which of your tents is lightest?
  -> The lightest tent is the **Summit Lite 2P** at **1.3 kg**.
  verdict: allow



[other] Can you write my history essay about Rome?
  -> I can't help with that here. Is there something else I can help you with?
  verdict: redirect



**What you see:** the product question is answered; the essay request is redirected by the responder site, while the
triage site kept observing. `immune.status()` lists each site with its profile and posture.

## 4. Rehearse an incident through your graph

`incident.forcedleak` reconstructs a reported attack on a CRM agent: a lead record tells the assistant to email data
to an outside address. Here it runs through a LangGraph agent, offline. The model's side is the recorded attack,
served by Immune's test harness, so no real model is asked to misbehave.

In [5]:
import tempfile
from pathlib import Path

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import immune
from immune.testing import FakeReply, FakeToolCall, ImmuneHarness, MockSensor, ScenarioLibrary

incident = ScenarioLibrary().find("incident.forcedleak")
turn = incident.turns[0]
recorded_email = turn.reply.tool_calls[0]
sent: list[dict] = []


@tool
def read_lead() -> str:
    """Read the newest lead from the CRM."""
    return turn.data[0].text


@tool
def send_email(to: str, subject: str = "", body: str = "") -> str:
    """Send an email."""
    sent.append({"to": to})
    return "sent"


harness = ImmuneHarness(
    Path(tempfile.mkdtemp()),
    sensor=MockSensor(incident.signals),
    script=[
        FakeReply(tool_calls=[FakeToolCall("read_lead", {})]),
        FakeReply(tool_calls=[FakeToolCall("send_email", dict(recorded_email.arguments))]),
        FakeReply("Done."),
    ],
    config={"sites": {"crm-agent": {"tools": {"send_email": {"egress": True}}}}},
)
TOOLS = [read_lead, send_email]
model = ChatOpenAI(model="gpt-5.4-mini", api_key="rehearsal", http_client=harness.http_client()).bind_tools(TOOLS)


def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")

with immune.site("crm-agent"):
    result = graph.compile().invoke({"messages": [("user", turn.user)]})

print("incident:", incident.title)
print("emails actually sent:", sent)
held = [message for message in result["messages"] if message.type == "ai" and harness.verdict(message)]
for message in held:
    verdict = harness.verdict(message)
    if verdict.hits:
        print("verdict:", verdict.action, "-", verdict.explanation)
harness.close()

incident: ForcedLeak: instructions in a web-to-lead form make the agent send CRM data out
emails actually sent: []
verdict: hold - hold: tool.destination_provenance


**What you see:** no email was sent. The address appeared only in the lead record, never in anything the user
said, so floor rule F4 (destination provenance) nominated the call, the recorded Jev answer confirmed it, and Immune
removed the call before LangGraph's tool node could run it.

## Common mistakes

| Avoid | Prefer instead | Why |
| --- | --- | --- |
| A new `immune.session()` id per turn | The thread id from your checkpointer config | Confirmations and taint follow the conversation |
| Leaving tool capabilities to guesswork | `sites.<site>.tools.<tool>: {egress, writes_state, irreversible, reads_private}` | Immune checks what each tool can do |
| Calling `immune.verdict()` inside a node right after `invoke` | `immune.verdict(message)` | LangGraph runs nodes in copies of the context |
| Wrapping tools to add your own confirmation prompts | Vaccines with `respond: {tool: confirm}` | Immune seals the confirmation to the exact call |

## Recap

- `immune.init()` screens every model call in a LangGraph graph; the graph doesn't change
- Declare tool capabilities and use your thread id as the session
- Risky calls become sealed confirmations on user-facing sites
- `immune.site()` inside a node gives each node its own posture

Next: [Reading verdicts](06_reading_verdicts.ipynb) · [Tools and agents](10_tools_and_agents.ipynb)